# 🧬 Pipeline scRNA-seq — IBD Colon/Recto - GSE214695

## Expresión diferencial pseudobulk (PyDESeq2)

**Flujo de trabajo:**
1. Cargar objeto anotado + metadatos de paciente (SraRunTable)
2. Verificar la fuente de conteos crudos
3. Exploración inicial (calidad de `cell_type_final`, cobertura por paciente)
4. Filtrado de tipos celulares
5. Agregación pseudobulk
6. DESeq2 por tipo celular, 3 comparaciones por pares
7. Extracción de resultados (DEG) y tablas de features para ML
8. Control de calidad.



# · Importaciones y configuración

In [ ]:
# Montar Google Drive y preparar el gestor de entornos Conda
from google.colab import drive
drive.mount('/content/drive')

!pip install -q condacolab
import condacolab
condacolab.install()

Mounted at /content/drive
⏬ Downloading https://github.com/conda-forge/miniforge/releases/download/25.11.0-1/Miniforge3-25.11.0-1-Linux-x86_64.sh...
📦 Installing...
📌 Adjusting configuration...
🩹 Patching environment...
⏲ Done in 0:00:09
🔁 Restarting kernel...


In [ ]:
# Instalación del entorno Conda propio de esta fase del proyecto (environment_deg_ml.yaml)

!conda env update -n base -f /content/drive/MyDrive/TFM_IBD_GSE214695/environment_deg_ml.yaml -q


Retrieving notices: ...working... done
Channels:
 - conda-forge
 - bioconda
 - defaults
Platform: linux-64
Solving environment: ...working... done
Preparing transaction: ...working... done
Verifying transaction: ...working... done
Executing transaction: ...working... done
Installing pip dependencies: ...working... done


In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import os
import numpy as np
import pandas as pd
import scipy.sparse as sp
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
import scanpy as sc
import yaml

from pydeseq2.dds import DeseqDataSet
from pydeseq2.ds import DeseqStats
from pydeseq2.default_inference import DefaultInference

sc.settings.verbosity = 1
print("Librerías cargadas")


Librerías cargadas


# · Configuración de rutas

In [ ]:
REPO_ROOT = Path("/content/drive/MyDrive/TFM_IBD_GSE214695")
with open(REPO_ROOT / "config" / "params.yaml") as f:
    PARAMS = yaml.safe_load(f)

PROJECT_ROOT = os.environ.get(PARAMS["project_root_env_var"], PARAMS["project_root_default"])

ANNOT_DIR   = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['07_annotated']}"
INPUT_PATH  = f"{ANNOT_DIR}/{PARAMS['outputs']['annotated_h5ad']}"

# ── Metadatos de paciente ─────
SRA_PATH = f"{PROJECT_ROOT}/{PARAMS['paths']['raw'].get('sra_run_table', 'data/raw/SraRunTable.csv')}"

OUTPUT_DIR = f"{PROJECT_ROOT}/{PARAMS['paths']['interim'].get('08_deg', 'data/interim/08_deg_pseudobulk')}"
Path(OUTPUT_DIR).mkdir(parents=True, exist_ok=True)

TABLES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['tables'].get('08_deg', 'reports/tables/08_deg_pseudobulk')}"
Path(TABLES_DIR).mkdir(parents=True, exist_ok=True)

FIGURES_DIR = f"{REPO_ROOT}/{PARAMS['paths']['reports']['figures'].get('08_deg', 'reports/figures/08_deg_pseudobulk')}"
Path(FIGURES_DIR).mkdir(parents=True, exist_ok=True)
sc.settings.figdir = FIGURES_DIR

# Ruta por si .raw no es válido
FILTERED_DIR = f"{PROJECT_ROOT}/{PARAMS['paths']['interim']['01_qc_filtered']}"

BATCH_KEY     = "sample_id"
CONDITION_KEY = "condition"
CELLTYPE_KEY  = "cell_type_final"
PALETTE       = {'HC': '#2E86AB', 'UC': '#E84855', 'CD': '#F4A261'}
CONDITIONS    = ['HC', 'UC', 'CD']
COMPARISONS   = [('UC', 'HC'), ('CD', 'HC'), ('UC', 'CD')]

# Umbrales empleados
MIN_CELLS_PER_PATIENT_CT = 10   # Crowell et al. 2020 / criterio de `muscat`
BORDERLINE_LOW, BORDERLINE_HIGH = 5, 10
MIN_REPLICATES_PER_GROUP = 2
LOW_POWER_REPLICATES     = 3
DEG_PADJ_MAX = 0.05
DEG_LOG2FC_MIN = 1.0

print("Configuración de rutas cargada")

Configuración de rutas cargada


# · Carga del objeto anotado y de los metadatos de paciente

El `sample_id` (ej.  `GSM6614348_HC-1`) se creó al inicio y no se ha modificado. El prefijo (`GSM6614348`) es el identificador GEO que coincide con la columna `Sample Name` del `SraRunTable.csv`. Ese archivo tiene varias filas por muestra; se colapsa a una fila por muestra tras verificar
que sexo, tejido y estado de la enfermedad son consistentes en todas las réplicas.


In [ ]:
adata = sc.read_h5ad(INPUT_PATH)
print(f"Objeto anotado cargado: {adata.n_obs:,} células × {adata.n_vars:,} genes")
print(f"Columnas de interés: "
      f"{[c for c in [BATCH_KEY, CONDITION_KEY, CELLTYPE_KEY] if c in adata.obs.columns]}")

assert BATCH_KEY in adata.obs.columns, f"Falta '{BATCH_KEY}' en .obs"
assert CONDITION_KEY in adata.obs.columns, f"Falta '{CONDITION_KEY}' en .obs"
assert CELLTYPE_KEY in adata.obs.columns, f"Falta '{CELLTYPE_KEY}' en .obs"

# ── Metadatos SRA ───────────────────────────────────────────────────────
df_sra = pd.read_csv(SRA_PATH, sep=';')
print(f"\nSraRunTable: {df_sra.shape[0]} filas × {df_sra.shape[1]} columnas")

# Colapsar a 1 fila por muestra y verificar consistencia
cols_needed = ['Sample Name', 'gender', 'tissue', 'disease_state']
missing_cols = [c for c in cols_needed if c not in df_sra.columns]
assert not missing_cols, f"Faltan columnas en SraRunTable: {missing_cols}"

n_inconsistent = (
    df_sra.groupby('Sample Name')[['gender', 'tissue', 'disease_state']]
    .nunique()
    .gt(1)
    .any(axis=1)
    .sum()
)
assert n_inconsistent == 0, (
    f"{n_inconsistent} muestras con valores inconsistentes")

df_meta = (df_sra.drop_duplicates(subset='Sample Name')
           .set_index('Sample Name')[['gender', 'tissue', 'disease_state']]
           .rename(columns={'gender': 'sexo', 'tissue': 'localizacion_biopsia'}))
print(f"   → Colapsado a {len(df_meta)} muestras únicas (GSM)")

# ── Enlazar con adata por el prefijo GSM ──────────────────
adata.obs['GSM'] = adata.obs[BATCH_KEY].astype(str).str.split('_').str[0]

n_gsm_adata = adata.obs['GSM'].nunique()
n_gsm_meta  = df_meta.index.nunique()
missing_gsm = set(adata.obs['GSM'].unique()) - set(df_meta.index)
print(f"\nGSM en adata: {n_gsm_adata} | GSM en metadatos: {n_gsm_meta} | sin metadatos: {len(missing_gsm)}")
assert not missing_gsm, f"Muestras sin metadatos SRA: {missing_gsm}"

adata.obs = adata.obs.join(df_meta, on='GSM')

# Verficación: disease_state del SRA debe coincidir con 'condition', calculada en notebook 03
map_disease = {'healthy control': 'HC', 'ulcerative colitis': 'UC', "Crohn's disease": 'CD'}
adata.obs['condition_from_sra'] = adata.obs['disease_state'].map(map_disease)
mismatch = (adata.obs['condition_from_sra'] != adata.obs[CONDITION_KEY]).sum()
print(f"Discrepancias: {mismatch}")
assert mismatch == 0, "El mapeo de condición no coincide con los metadatos SRA."

# patient_id: no existe una columna dedicada en ningún notebook previo.

adata.obs['patient_id'] = adata.obs[BATCH_KEY].astype(str)

print(f"\n Metadatos enlazados correctamente.")
print(adata.obs[['patient_id', CONDITION_KEY, 'sexo', 'localizacion_biopsia']]
      .drop_duplicates().sort_values('patient_id').to_string(index=False))


Objeto anotado cargado: 43,388 células × 33,538 genes
Columnas de interés: ['sample_id', 'condition', 'cell_type_final']

SraRunTable: 49 filas × 33 columnas
   → Colapsado a 18 muestras únicas (GSM)

GSM en adata: 18 | GSM en metadatos: 18 | sin metadatos: 0
Discrepancias: 0

 Metadatos enlazados correctamente.
     patient_id condition   sexo localizacion_biopsia
GSM6614348_HC-1        HC   male                sigma
GSM6614349_HC-2        HC   male                sigma
GSM6614350_HC-3        HC female                sigma
GSM6614351_HC-4        HC   male                sigma
GSM6614352_HC-5        HC   male                sigma
GSM6614353_HC-6        HC female                sigma
GSM6614354_UC-1        UC   male                sigma
GSM6614355_UC-2        UC   male     transverse colon
GSM6614356_UC-3        UC female               rectum
GSM6614357_UC-4        UC female                sigma
GSM6614358_UC-5        UC   male               rectum
GSM6614359_UC-6        UC female      

# · Estructura experimental y diseño


In [ ]:
# ── Tabla de contingencia condición × localización de biopsia ──────────
df_patients = adata.obs[['patient_id', CONDITION_KEY, 'sexo', 'localizacion_biopsia']].drop_duplicates()

print("Distribución condición × localización de biopsia:")
ct_tissue = pd.crosstab(df_patients[CONDITION_KEY], df_patients['localizacion_biopsia'])
print(ct_tissue.to_string())

print("Distribución condición × sexo:")
ct_sexo = pd.crosstab(df_patients[CONDITION_KEY], df_patients['sexo'])
print(ct_sexo.to_string())

# ── Verificación de rango: ~condition + localizacion_biopsia (subconjunto HC) ──
def check_design_rank(df, formula_cols):
    # Devuelve (rango, n_columnas) de la matriz de diseño dummy-encoded.
    X = pd.get_dummies(df[formula_cols], drop_first=True).astype(float)
    X.insert(0, 'Intercept', 1.0)
    return np.linalg.matrix_rank(X.to_numpy(dtype=float)), X.shape[1]

# Dentro de HC la localizacion_biopsia no varía
hc_only = df_patients[df_patients[CONDITION_KEY] == 'HC']
print(f"\nVerificación variabilidad de localizacion_biopsia en el grupo HC → "
      f"Número de valores únicos en HC: {hc_only['localizacion_biopsia'].nunique()}"
      f"{'Todos los HC comparten la misma ubicación, se excluye del diseño para eviar colinealidad' if hc_only['localizacion_biopsia'].nunique() == 1 else 'Variable válida'}")

rank_full, ncols_full = check_design_rank(df_patients, [CONDITION_KEY, 'localizacion_biopsia'])
print(f"Rango de la matriz condition + localizacion_biopsia (todas las muestras) → "
      f"Rango: {rank_full} de {ncols_full} columnas → "
      f"{'RANGO COMPLETO' if rank_full == ncols_full else 'RANGO DEFICIENTE'}")

rank_sexo, ncols_sexo = check_design_rank(df_patients, [CONDITION_KEY, 'sexo'])
print(f"Rango de la matriz condition + sexo (todas las muestras) → "
      f"Rango: {rank_sexo} de {ncols_sexo} columnas → "
      f"{'RANGO COMPLETO' if rank_sexo == ncols_sexo else 'RANGO DEFICIENTE'}")

DECISION_INCLUDE_SEXO_GLOBAL = (rank_sexo == ncols_sexo)
print(f"\n Localizacion_biopsia excluida de todos los diseños. "
      f"Sexo se evaluará por tipo celular posteriormente")


Distribución condición × localización de biopsia:
localizacion_biopsia  ascending colon  descending colon  rectum  sigma  transverse colon
condition                                                                               
CD                                  1                 1       0      4                 0
HC                                  0                 0       0      6                 0
UC                                  0                 0       3      2                 1
Distribución condición × sexo:
sexo       female  male
condition              
CD              2     4
HC              2     4
UC              3     3

Verificación variabilidad de localizacion_biopsia en el grupo HC → Número de valores únicos en HC: 1Todos los HC comparten la misma ubicación, se excluye del diseño para eviar colinealidad
Rango de la matriz condition + localizacion_biopsia (todas las muestras) → Rango: 7 de 7 columnas → RANGO COMPLETO
Rango de la matriz condition + sexo (todas las mu

# · Verificación de la fuente de conteos crudos



In [ ]:
def is_valid_raw_counts(X):
    # Comprueba enteros no negativos, sobre `.data` si la matriz es sparse.
    vals = X.data if sp.issparse(X) else X
    if vals.size == 0:
        return False
    are_integers = np.allclose(vals, np.round(vals))
    are_non_negative = (vals >= 0).all()
    return bool(are_integers and are_non_negative)

raw_source = None

# A partir de aquí, RAW_COUNTS_MATRIX / RAW_COUNTS_GENES quedan siempre alineados
# con adata.obs_names (mismo orden de células), sea cual sea la fuente elegida.

if adata.raw is not None and is_valid_raw_counts(adata.raw.X):
    raw_source = 'adata.raw.X'
    RAW_COUNTS_MATRIX = adata.raw.X
    RAW_COUNTS_GENES  = adata.raw.var_names
    print(f" '{raw_source}' contiene conteos enteros no negativos. Se usa como fuente de conteos crudos.")
elif is_valid_raw_counts(adata.X):
    raw_source = 'adata.X'
    RAW_COUNTS_MATRIX = adata.X
    RAW_COUNTS_GENES  = adata.var_names
    print(f"  adata.raw no es válido, pero 'adata.X' sí contiene conteos enteros. Se usa como fallback.")
else:
    print("  Ni adata.raw.X ni adata.X son conteos enteros no negativos.")
    print(f"   Releyendo conteos crudos originales desde {FILTERED_DIR} ...")
    import glob
    filt_files = sorted(glob.glob(f"{FILTERED_DIR}/*_filtered.h5ad"))
    assert filt_files, f"No se encontraron archivos filtrados en {FILTERED_DIR}"
    raw_pieces = []
    for f in filt_files:
        tmp = sc.read_h5ad(f)
        raw_pieces.append(tmp)
    import anndata as ad
    adata_raw_reload = ad.concat(raw_pieces, join='outer', fill_value=0)

    if not is_valid_raw_counts(adata_raw_reload.X):
        raise RuntimeError(
            "Ninguna de las tres fuentes contiene conteos enteros no negativos.")

    # Alinear barcodes con adata.obs_names.
    # Detener ejecución si falta alguno para evitar alineaciones parciales entre notebooks.

    missing_barcodes = adata.obs_names.difference(adata_raw_reload.obs_names)
    assert len(missing_barcodes) == 0, (
        f"{len(missing_barcodes)} barcodes de 07_annotated no se encuentran en 01_qc_filtered. Deteniendo."
    )
    adata_raw_reload = adata_raw_reload[adata.obs_names].copy()

    raw_source = '01_qc_filtered (releído)'
    RAW_COUNTS_MATRIX = adata_raw_reload.X
    RAW_COUNTS_GENES  = adata_raw_reload.var_names
    print("Conteos enteros confirmados en 01_qc_filtered, alineados a adata.obs_names.")

print(f"\nFuente de conteos crudos usada para el pseudobulk: {raw_source}")
print(f"Matriz de conteos: {RAW_COUNTS_MATRIX.shape[0]:,} células × {len(RAW_COUNTS_GENES):,} genes")


 'adata.raw.X' contiene conteos enteros no negativos. Se usa como fuente de conteos crudos.

Fuente de conteos crudos usada para el pseudobulk: adata.raw.X
Matriz de conteos: 43,388 células × 33,538 genes


# · Exploración inicial

In [ ]:
# ── Categorías especiales dentro de cell_type_final ─────────────────────
vc_final = adata.obs[CELLTYPE_KEY].value_counts()
special_mask = vc_final.index.str.contains('Doublet|Unresolved|heredado', case=False, regex=True)
print("Categorías especiales en cell_type_final:")
print(vc_final[special_mask].to_string())

n_doublet    = (adata.obs[CELLTYPE_KEY] == 'Doublet').sum()
n_unresolved = (adata.obs[CELLTYPE_KEY] == 'Unresolved').sum()
print(f"\n'Doublet': {n_doublet:,} células")
print(f"'Unresolved': {n_unresolved:,} células")

# ── Células por (paciente, condición, tipo celular) ─────────────────────
coverage = (adata.obs.groupby(['patient_id', CONDITION_KEY, CELLTYPE_KEY], observed=True)
            .size().rename('n_cells').reset_index())
print(f"\nCombinaciones (paciente, condición, tipo celular): {len(coverage)}")
print(f"Mediana de células por combinación: {coverage['n_cells'].median():.0f}")


Categorías especiales en cell_type_final:
cell_type_final
Epithelium Ribhi (heredado de r0.5-C3)                   4673
Unresolved                                                505
PC IgG 1 (heredado de r0.5-C4, doublet score elevado)     115
Doublet                                                   111

'Doublet': 111 células
'Unresolved': 505 células

Combinaciones (paciente, condición, tipo celular): 329
Mediana de células por combinación: 48


In [ ]:
# ── Distribución de covariables por condición ─
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

ct_tissue.plot(kind='bar', stacked=True, ax=axes[0],
               color=sns.color_palette('Set2', ct_tissue.shape[1]))
axes[0].set_title("Localización de biopsia por condición")
axes[0].set_ylabel("Nº pacientes")
axes[0].legend(fontsize=7, title='Localización')

ct_sexo.plot(kind='bar', stacked=True, ax=axes[1],
             color=sns.color_palette('Set2', ct_sexo.shape[1]))
axes[1].set_title("Sexo por condición")
axes[1].set_ylabel("Nº pacientes")
axes[1].legend(fontsize=7, title='Sexo')

plt.tight_layout()
fig.savefig(f"{FIGURES_DIR}/covariates_by_condition.png", dpi=130, bbox_inches='tight', facecolor='white')
plt.close(fig)
print("   → Figura guardada: covariates_by_condition.png")


   → Figura guardada: covariates_by_condition.png


# · Filtrado de tipos celulares



In [ ]:
# ── 'Doublet' ──
doublet_mask = adata.obs[CELLTYPE_KEY] == 'Doublet'
if 'doublet_score' in adata.obs.columns and doublet_mask.sum() > 0:
    ds_doublet = adata.obs.loc[doublet_mask, 'doublet_score']
    ds_global  = adata.obs['doublet_score']
    print("Distribución de doublet_score en células etiquetadas 'Doublet':")
    print(ds_doublet.describe().to_string())
    print(f"\ndoublet_score global: media={ds_global.mean():.3f}")

    q25, q75 = ds_doublet.quantile([0.25, 0.75])
    n_borderline = (ds_doublet <= q25).sum()
    print(f"\nCélulas en el cuartil menos extremo del grupo 'Doublet' "
          f"(score ≤ P25={q25:.3f}, frente a min={ds_doublet.min():.3f}, "
          f"max={ds_doublet.max():.3f}): {n_borderline} de {doublet_mask.sum()}")

else:
    print("  No se encontró 'doublet_score' en .obs")
    n_borderline = 0

# Documentado: se excluyen en bloque, no rescatable con la información disponible.
print(f"\n Decisión 'Doublet': excluir las {doublet_mask.sum()} células. ")


Distribución de doublet_score en células etiquetadas 'Doublet':
count    111.000000
mean       0.243472
std        0.126979
min        0.031000
25%        0.121438
50%        0.256724
75%        0.362416
max        0.504348

doublet_score global: media=0.054

Células en el cuartil menos extremo del grupo 'Doublet' (score ≤ P25=0.121, frente a min=0.031, max=0.504): 28 de 111

 Decisión 'Doublet': excluir las 111 células. 


In [ ]:
# ── 'Unresolved' ──
unresolved_mask = adata.obs[CELLTYPE_KEY] == 'Unresolved'

markers_canonical = {
    "Epitelial":   ["EPCAM", "KRT20", "MUC2"],
    "Goblet":      ["MUC2", "TFF3"],
    "T cell":      ["CD3D", "CD3E", "CD8A", "CD4"],
    "B cell":      ["CD79A", "MS4A1"],
    "Plasma":      ["IGHA1", "MZB1"],
    "Macrophage":  ["C1QA", "CD68"],
    "Monocyte":    ["LYZ", "S100A8"],
    "Fibroblast":  ["COL1A1", "DCN"],
    "Endothelial": ["VWF", "PECAM1"],
    "Mast":        ["TPSAB1", "CPA3"],
    "Tuft":        ["POU2F3", "TRPM5"],
}
scores = {}
for lineage, genes in markers_canonical.items():
    genes_present = [g for g in genes if g in adata.var_names]
    if not genes_present:
        continue
    sc.tl.score_genes(adata, gene_list=genes_present, score_name=f"score_{lineage}", use_raw=False)
    scores[lineage] = f"score_{lineage}"

score_cols = list(scores.values())
mean_scores_unresolved = adata.obs.loc[unresolved_mask, score_cols].mean().sort_values(ascending=False)
mean_scores_global      = adata.obs[score_cols].mean()

print("Score medio de marcadores canónicos en células 'Unresolved' (vs. media global):")
comp = pd.DataFrame({'unresolved': mean_scores_unresolved,
                      'global': mean_scores_global.reindex(mean_scores_unresolved.index)})
print(comp.to_string())

top_lineage = mean_scores_unresolved.index[0]
top_score_unresolved = mean_scores_unresolved.iloc[0]
top_score_global = mean_scores_global[top_lineage]
margin_2nd = mean_scores_unresolved.iloc[0] - mean_scores_unresolved.iloc[1]
lineage_name = top_lineage.replace('score_', '')

# Comparación de cuánto se acerca el score de 'Unresolved' al nivel típico de
# ese mismo linaje en células ya anotadas como tal.

pct_of_global = (top_score_unresolved / top_score_global * 100) if top_score_global > 0 else float('nan')
WEAK_SIGNAL_THRESHOLD_PCT = 60  # por debajo se considera señal parcial/no concluyente

print(f"\nLinaje dominante dentro de 'Unresolved': {lineage_name} "
      f"({top_score_unresolved:.3f}, margen sobre el 2º linaje: {margin_2nd:.3f})")
print(f"Score típico de {lineage_name} en células ya anotadas como tal: {top_score_global:.3f}")
print(f"→ 'Unresolved' alcanza solo el {pct_of_global:.0f}% del score típico de {lineage_name}")

if pct_of_global < WEAK_SIGNAL_THRESHOLD_PCT:
    print(f"\n Decisión 'Unresolved': excluir las {unresolved_mask.sum()} células en bloque.")
else:
    print(f"\n  Señal hacia {lineage_name} relativamente fuerte ({pct_of_global:.0f}% del "
          f"nivel típico) — revisar.")

Score medio de marcadores canónicos en células 'Unresolved' (vs. media global):
                   unresolved    global
score_Plasma         0.339512  0.924944
score_T cell         0.001386  0.101707
score_Tuft          -0.007875  0.000091
score_Mast          -0.032732  0.002741
score_Macrophage    -0.032856 -0.002733
score_Endothelial   -0.037904 -0.010401
score_B cell        -0.047475 -0.104418
score_Fibroblast    -0.092887 -0.028612
score_Monocyte      -0.169396 -0.183899
score_Goblet        -0.250562 -0.072148
score_Epitelial     -0.287666 -0.141511

Linaje dominante dentro de 'Unresolved': Plasma (0.340, margen sobre el 2º linaje: 0.338)
Score típico de Plasma en células ya anotadas como tal: 0.925
→ 'Unresolved' alcanza solo el 37% del score típico de Plasma

 Decisión 'Unresolved': excluir las 505 células en bloque.


**Nota**: las etiquetas con sufijo entre paréntesis (ej. "... heredado de ...") se limpian y se quedan en el análisis pseudobulk, agrupadas solo por el nombre del tipo celular base (sin el sufijo).

In [ ]:
# ── Aplicar exclusión ────────
# adata original permanece intacto.
pb_mask = ~adata.obs[CELLTYPE_KEY].isin(['Doublet', 'Unresolved'])
adata_pb = adata[pb_mask].copy()

RAW_COUNTS_MATRIX_PB = RAW_COUNTS_MATRIX[pb_mask.values]
if sp.issparse(RAW_COUNTS_MATRIX_PB):
    RAW_COUNTS_MATRIX_PB = RAW_COUNTS_MATRIX_PB.tocsr()

# Limpiar el sufijo "(heredado de r0.5-CX...)" para agrupar bajo el tipo celular base.
adata_pb.obs['cell_type_pb'] = (
    adata_pb.obs[CELLTYPE_KEY].astype(str).str.replace(r"\s*\(heredado.*\)", "", regex=True)
)
adata_pb.obs['was_inherited'] = adata_pb.obs[CELLTYPE_KEY].astype(str).str.contains('heredado')

print(f"Células antes de excluir Doublet/Unresolved: {adata.n_obs:,}")
print(f"Células tras excluir: {adata_pb.n_obs:,} "
      f"({100*(adata.n_obs - adata_pb.n_obs)/adata.n_obs:.1f}% excluido)")
print(f"Tipos celulares (cell_type_pb) tras limpiar sufijos: {adata_pb.obs['cell_type_pb'].nunique()}")
assert RAW_COUNTS_MATRIX_PB.shape[0] == adata_pb.n_obs, (
    "Desalineación entre RAW_COUNTS_MATRIX_PB y adata_pb."
)


Células antes de excluir Doublet/Unresolved: 43,388
Células tras excluir: 42,772 (1.4% excluido)
Tipos celulares (cell_type_pb) tras limpiar sufijos: 17


# · Umbral mínimo de células

In [ ]:
cov_pb = (adata_pb.obs.groupby(['patient_id', CONDITION_KEY, 'cell_type_pb'], observed=True)
          .size().rename('n_cells').reset_index())

n_below_min      = (cov_pb['n_cells'] < MIN_CELLS_PER_PATIENT_CT).sum()
n_borderline_zone = cov_pb['n_cells'].between(BORDERLINE_LOW, BORDERLINE_HIGH - 1).sum()
print(f"Combinaciones (paciente, tipo celular) por debajo de {MIN_CELLS_PER_PATIENT_CT} células: {n_below_min}")
print(f"Combinaciones en zona límite [{BORDERLINE_LOW}, {BORDERLINE_HIGH}) células: {n_borderline_zone}")
if n_borderline_zone > 0:
    print(cov_pb[cov_pb['n_cells'].between(BORDERLINE_LOW, BORDERLINE_HIGH - 1)]
          .sort_values('n_cells').to_string(index=False))
    print("   → Zona límite reportada; se mantiene el umbral estándar de 10.")

cov_pb_valid = cov_pb[cov_pb['n_cells'] >= MIN_CELLS_PER_PATIENT_CT].copy()

# ── Tipos celulares presentes en una sola condición → excluir de comparaciones ──
patients_per_ct = cov_pb_valid.groupby('cell_type_pb')[CONDITION_KEY].nunique()
single_condition_ct = patients_per_ct[patients_per_ct < 2].index.tolist()
multi_condition_ct  = patients_per_ct[patients_per_ct >= 2].index.tolist()

if single_condition_ct:
    print(f"\n  Tipos celulares presentes en una sola condición: {single_condition_ct}")

print(f"\n Tipos celulares que entran en el análisis DEG: {len(multi_condition_ct)}")
for ct in multi_condition_ct:
    print(f"   - {ct}")


Combinaciones (paciente, tipo celular) por debajo de 10 células: 44
Combinaciones en zona límite [5, 10) células: 25
     patient_id condition cell_type_pb  n_cells
GSM6614348_HC-1        HC     PC IgG 1        5
GSM6614349_HC-2        HC  Endothelium        5
GSM6614351_HC-4        HC       B cell        5
GSM6614350_HC-3        HC     PC IgG 1        5
GSM6614357_UC-4        UC   Tuft cells        5
GSM6614357_UC-4        UC     PC IgG 1        5
GSM6614355_UC-2        UC         Glia        5
GSM6614364_CD-5        CD     PC IgG 1        5
GSM6614362_CD-3        CD     PC IgG 1        5
GSM6614360_CD-1        CD           N1        6
GSM6614355_UC-2        UC   Tuft cells        6
GSM6614348_HC-1        HC         Glia        6
GSM6614362_CD-3        CD         Glia        7
GSM6614353_HC-6        HC         Glia        7
GSM6614350_HC-3        HC       PC IgG        7
GSM6614354_UC-1        UC  Colonocytes        7
GSM6614364_CD-5        CD   Tuft cells        7
GSM6614365_CD-6    

# · Agregación pseudobulk


In [ ]:
X_raw = RAW_COUNTS_MATRIX_PB
genes_raw = RAW_COUNTS_GENES
if sp.issparse(X_raw):
    X_raw = X_raw.tocsr()

pseudobulk = {}  # cell_type -> (counts_df [samples x genes], meta_df)

for ct in multi_condition_ct:
    valid_patients_ct = cov_pb_valid.loc[cov_pb_valid['cell_type_pb'] == ct, 'patient_id']
    mask_ct = (adata_pb.obs['cell_type_pb'] == ct) & (adata_pb.obs['patient_id'].isin(valid_patients_ct))

    rows = []
    sample_ids = []
    meta_rows = []
    for pid, grp in adata_pb.obs.loc[mask_ct].groupby('patient_id', observed=True):
        idx = adata_pb.obs.index.get_indexer(grp.index)
        summed = np.asarray(X_raw[idx].sum(axis=0)).flatten()
        rows.append(summed)
        sample_ids.append(pid)
        meta_rows.append({
            'patient_id': pid,
            CONDITION_KEY: grp[CONDITION_KEY].iloc[0],
            'sexo': grp['sexo'].iloc[0],
            'n_cells_aggregated': len(grp),
        })

    counts_df = pd.DataFrame(rows, index=sample_ids, columns=genes_raw)
    meta_df   = pd.DataFrame(meta_rows).set_index('patient_id')

    # Eliminar genes con suma total 0 en todas las muestras de este tipo celular
    genes_nonzero = counts_df.sum(axis=0) > 0
    counts_df = counts_df.loc[:, genes_nonzero]

    pseudobulk[ct] = (counts_df, meta_df)
    print(f"{ct:35s}: {counts_df.shape[0]} muestras × {counts_df.shape[1]:,} genes "
          f"(de {len(genes_nonzero):,} originales, {genes_nonzero.sum():,} con suma>0)")

print(f"\n Pseudobulk generado para {len(pseudobulk)} tipos celulares.")


B cell                             : 16 muestras × 18,615 genes (de 33,538 originales, 18,615 con suma>0)
CD4                                : 18 muestras × 20,837 genes (de 33,538 originales, 20,837 con suma>0)
CD8                                : 18 muestras × 19,641 genes (de 33,538 originales, 19,641 con suma>0)
Colonocytes                        : 17 muestras × 18,630 genes (de 33,538 originales, 18,630 con suma>0)
Endothelium                        : 15 muestras × 15,222 genes (de 33,538 originales, 15,222 con suma>0)
Epithelium Ribhi                   : 18 muestras × 21,426 genes (de 33,538 originales, 21,426 con suma>0)
Fibroblast (subtipo indeterminado) : 18 muestras × 20,929 genes (de 33,538 originales, 20,929 con suma>0)
Glia                               : 7 muestras × 12,967 genes (de 33,538 originales, 12,967 con suma>0)
Goblet                             : 15 muestras × 19,861 genes (de 33,538 originales, 19,861 con suma>0)
Mast                               : 18 muestra

# 🔬 DESeq2 por tipo celular



In [ ]:
def design_is_full_rank(meta_df, cols):
    X = pd.get_dummies(meta_df[cols], drop_first=True).astype(float)
    X.insert(0, 'Intercept', 1.0)
    return np.linalg.matrix_rank(X.to_numpy(dtype=float)) == X.shape[1]

inference = DefaultInference(n_cpus=4)

dds_by_ct = {}
design_by_ct = {}

for ct, (counts_df, meta_df) in pseudobulk.items():
    n_per_group = meta_df[CONDITION_KEY].value_counts()
    groups_present = n_per_group.index.tolist()

    use_sexo = (
        meta_df['sexo'].nunique() > 1
        and design_is_full_rank(meta_df, [CONDITION_KEY, 'sexo'])
    )
    design = "~ sexo + condition" if use_sexo else "~ condition"
    design_by_ct[ct] = design

    print(f"\n--- {ct} ---")
    print(f"   Réplicas por grupo: {n_per_group.to_dict()}")
    print(f"   Diseño: {design}")

    if (n_per_group < MIN_REPLICATES_PER_GROUP).any():
        print(f"     SALTADO: al menos un grupo tiene <{MIN_REPLICATES_PER_GROUP} réplicas.")
        continue
    if (n_per_group < LOW_POWER_REPLICATES).any():
        print(f"     AVISO: algún grupo tiene {n_per_group.min()} réplicas (2-3) — baja potencia estadística.")

    counts_int = counts_df.round().astype(int)
    dds = DeseqDataSet(
        counts=counts_int,
        metadata=meta_df,
        design=design,
        refit_cooks=True,
        inference=inference,
    )
    dds.deseq2()
    dds_by_ct[ct] = dds
    print(f"  ✅ DESeq2 ajustado ({counts_int.shape[0]} muestras × {counts_int.shape[1]:,} genes)")



--- B cell ---
   Réplicas por grupo: {'UC': 6, 'CD': 6, 'HC': 4}
   Diseño: ~ sexo + condition
Using None as control genes, passed at DeseqDataSet initialization


Fitting size factors...
... done in 0.02 seconds.

Fitting dispersions...
... done in 23.38 seconds.

Fitting dispersion trend curve...
... done in 0.41 seconds.

Fitting MAP dispersions...
... done in 34.60 seconds.

Fitting LFCs...
... done in 36.62 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 0 outlier genes.

Fitting size factors...
... done in 0.02 seconds.



   ✅ DESeq2 ajustado (16 muestras × 18,615 genes)

--- CD4 ---
   Réplicas por grupo: {'HC': 6, 'UC': 6, 'CD': 6}
   Diseño: ~ sexo + condition
Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 29.51 seconds.

Fitting dispersion trend curve...
... done in 0.57 seconds.

Fitting MAP dispersions...
... done in 35.49 seconds.

Fitting LFCs...
... done in 37.75 seconds.

Calculating cook's distance...
... done in 0.05 seconds.

Replacing 0 outlier genes.

Fitting size factors...
... done in 0.02 seconds.



   ✅ DESeq2 ajustado (18 muestras × 20,837 genes)

--- CD8 ---
   Réplicas por grupo: {'HC': 6, 'UC': 6, 'CD': 6}
   Diseño: ~ sexo + condition
Using None as control genes, passed at DeseqDataSet initialization


...

Fitting dispersions...
... done in 23.97 seconds.

Fitting dispersion trend curve...
... done in 0.64 seconds.

Fitting MAP dispersions...
... done in 30.54 seconds.

Fitting LFCs...
... done in 34.56 seconds.

Calculating cook's distance...
... done in 0.03 seconds.

Replacing 0 outlier genes.

Fitting size factors...
... done in 0.01 seconds.



   ✅ DESeq2 ajustado (17 muestras × 18,675 genes)

--- PC IgG 1 ---
   Réplicas por grupo: {'HC': 1, 'UC': 1}
   Diseño: ~ condition
   ⚠️  SALTADO: al menos un grupo tiene <2 réplicas.

--- Plasma (subtipo indeterminado) ---
   Réplicas por grupo: {'HC': 6, 'UC': 6, 'CD': 6}
   Diseño: ~ sexo + condition
Using None as control genes, passed at DeseqDataSet initialization


Fitting dispersions...
... done in 17.79 seconds.

Fitting dispersion trend curve...
... done in 0.37 seconds.

Fitting MAP dispersions...
... done in 30.14 seconds.

Fitting LFCs...


   ✅ DESeq2 ajustado (18 muestras × 15,577 genes)

--- Tuft cells ---
   Réplicas por grupo: {'HC': 6, 'CD': 4, 'UC': 1}
   Diseño: ~ sexo + condition
   ⚠️  SALTADO: al menos un grupo tiene <2 réplicas.


... done in 30.98 seconds.

Calculating cook's distance...
... done in 0.04 seconds.

Replacing 0 outlier genes.



In [ ]:
stats_by_ct_comparison = {}  # (cell_type, 'tested_vs_ref') -> DeseqStats

for ct, dds in dds_by_ct.items():
    groups_present = set(dds.obs[CONDITION_KEY].unique())
    for tested, ref in COMPARISONS:
        if tested not in groups_present or ref not in groups_present:
            print(f"{ct:30s} | {tested} vs {ref}: SALTADO (grupo ausente para este tipo celular)")
            continue
        n_tested = (dds.obs[CONDITION_KEY] == tested).sum()
        n_ref    = (dds.obs[CONDITION_KEY] == ref).sum()
        if n_tested < MIN_REPLICATES_PER_GROUP or n_ref < MIN_REPLICATES_PER_GROUP:
            print(f"{ct:30s} | {tested} vs {ref}: SALTADO (<2 réplicas en algún grupo)")
            continue

        ds = DeseqStats(dds, contrast=[CONDITION_KEY, tested, ref], inference=inference)
        ds.summary()
        stats_by_ct_comparison[(ct, f"{tested}_vs_{ref}")] = ds

        low_power = "  baja potencia (2-3 réplicas)" if min(n_tested, n_ref) <= LOW_POWER_REPLICATES else ""
        print(f"{ct:30s} | {tested} vs {ref}: n={n_tested}/{n_ref}{low_power}")

print(f"\n✅ {len(stats_by_ct_comparison)} comparaciones completadas.")


Running Wald tests...
... done in 6.27 seconds.



Log2 fold change & Wald test p-value: condition UC vs HC
            baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1  0.117578        0.900422  2.594682  0.347026  0.728572  0.999696
AL669831.2  0.020517        0.878735  4.489424  0.195734  0.844818  0.999696
AL669831.5  2.379829       -0.245811  0.954618 -0.257496  0.796796  0.999696
LINC00115   1.074299       -1.401136  1.263669 -1.108784  0.267523  0.999696
FAM41C      1.621132        0.167662  1.127398  0.148716  0.881778  0.999696
...              ...             ...       ...       ...       ...       ...
AC007325.4  0.143208        0.887254  2.807518  0.316028  0.751981  0.999696
AL354822.1  1.257324        0.416836  1.395440  0.298713  0.765159  0.999696
AC004556.1  0.139908        1.175532  3.388992  0.346868  0.728691  0.999696
AC233755.1  0.110788        0.594322  4.477923  0.132723  0.894413  0.999696
AC240274.1  0.423320       -0.498058  2.361675 -0.210892  0.832972  0.999696

[18615 rows x 6 co

Running Wald tests...
... done in 5.22 seconds.



Log2 fold change & Wald test p-value: condition CD vs HC
            baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1  0.117578        0.633543  2.374826  0.266775  0.789643  0.999917
AL669831.2  0.020517        0.047070  4.315867  0.010906  0.991298  0.999917
AL669831.5  2.379829        0.817934  0.825720  0.990571  0.321895  0.999917
LINC00115   1.074299       -1.113160  1.117621 -0.996008  0.319246  0.999917
FAM41C      1.621132       -1.297380  1.111704 -1.167019  0.243203  0.999917
...              ...             ...       ...       ...       ...       ...
AC007325.4  0.143208        0.798100  2.572531  0.310239  0.756379  0.999917
AL354822.1  1.257324       -0.711996  1.349004 -0.527794  0.597642  0.999917
AC004556.1  0.139908        0.241677  3.248346  0.074400  0.940692  0.999917
AC233755.1  0.110788        1.199921  4.191166  0.286298  0.774650  0.999917
AC240274.1  0.423320        0.096237  2.123720  0.045315  0.963856  0.999917

[18615 rows x 6 co

Running Wald tests...
... done in 7.77 seconds.



...

Running Wald tests...
... done in 5.29 seconds.



Log2 fold change & Wald test p-value: condition CD vs HC
            baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1  0.042392       -0.275973  3.986334 -0.069230  0.944807  0.996505
AL669831.5  0.798994        2.924816  2.137607  1.368266  0.171229  0.996505
LINC00115   0.495924       -1.356806  1.599679 -0.848174  0.396341  0.996505
FAM41C      0.060361        0.733024  3.992433  0.183603  0.854325  0.996505
AL645608.3  0.039576       -0.320210  3.981096 -0.080433  0.935893  0.996505
...              ...             ...       ...       ...       ...       ...
AC007325.4  0.171133       -0.571723  3.035307 -0.188358  0.850596  0.996505
AL354822.1  0.185557       -0.076280  2.025709 -0.037656  0.969962  0.996505
AC233755.2  0.230877       -0.253186  2.990855 -0.084653  0.932537  0.996505
AC233755.1  0.117539       -0.218838  2.615068 -0.083684  0.933308  0.996505
AC240274.1  0.073805        0.733024  3.992433  0.183603  0.854325  0.996505

[15577 rows x 6 co

Running Wald tests...


Log2 fold change & Wald test p-value: condition UC vs CD
            baseMean  log2FoldChange     lfcSE      stat    pvalue      padj
AL627309.1  0.042392        0.687135  4.098277  0.167664  0.866847  0.998984
AL669831.5  0.798994       -2.783494  2.348650 -1.185146  0.235960  0.998984
LINC00115   0.495924        0.085412  1.904932  0.044838  0.964237  0.998984
FAM41C      0.060361        0.317881  4.024217  0.078992  0.937039  0.998984
AL645608.3  0.039576        0.806087  4.090137  0.197081  0.843764  0.998984
...              ...             ...       ...       ...       ...       ...
AC007325.4  0.171133        1.658910  3.030343  0.547433  0.584081  0.998984
AL354822.1  0.185557        0.825849  2.106011  0.392139  0.694956  0.998984
AC233755.2  0.230877       -0.705391  3.263219 -0.216164  0.828860  0.998984
AC233755.1  0.117539        1.479006  2.570190  0.575446  0.564989  0.998984
AC240274.1  0.073805        0.317881  4.024217  0.078992  0.937039  0.998984

[15577 rows x 6 co

... done in 4.40 seconds.



# · Extracción de resultados (DEG)

In [ ]:
deg_tables = {}  # (cell_type, comparison) -> DataFrame filtrado

for (ct, comparison), ds in stats_by_ct_comparison.items():
    res = ds.results_df.copy()
    res.index.name = 'gene'
    res = res.reset_index()
    res['cell_type']  = ct
    res['comparison'] = comparison

    sig = res[(res['padj'] < DEG_PADJ_MAX) & (res['log2FoldChange'].abs() > DEG_LOG2FC_MIN)].copy()
    deg_tables[(ct, comparison)] = sig

    ct_safe = ct.replace(' ', '_').replace('/', '-')
    out_path = f"{TABLES_DIR}/DEG_{ct_safe}_{comparison}.csv"
    res.to_csv(out_path, index=False, float_format='%.5g')
    print(f"{ct:30s} | {comparison:12s}: {len(sig):5d} genes significativos "
          f"(de {len(res):,} testados) → {Path(out_path).name}")

all_deg = pd.concat(deg_tables.values(), ignore_index=True) if deg_tables else pd.DataFrame()
print(f"\n Total genes-comparación significativos (padj<{DEG_PADJ_MAX}, |log2FC|>{DEG_LOG2FC_MIN}): {len(all_deg):,}")


In [ ]:
# Resumen descriptivo para la discusion

meta_summary = pd.read_csv(f"{TABLES_DIR}/features_wide_table_metadata.csv")

print("Total DEGs:", len(meta_summary))

por_tipo = meta_summary.groupby("tipo_celular").size().sort_values(ascending=False)
print("\nTipo celular con mas DEGs:")
print(por_tipo.head(5))

top_tipo = por_tipo.index[0]
print(f"\nDesglose por comparacion dentro de {top_tipo}:")
print(meta_summary[meta_summary["tipo_celular"] == top_tipo].groupby("comparacion").size().sort_values(ascending=False))

for comp in ["UC_vs_HC", "CD_vs_HC"]:
    sub = meta_summary[meta_summary["comparacion"] == comp]
    print(f"\n--- {comp} ---")
    print("Mas sobreexpresado:", sub.loc[sub["log2FC"].idxmax(), ["gen", "tipo_celular", "log2FC"]].to_dict())
    print("Mas infraexpresado:", sub.loc[sub["log2FC"].idxmin(), ["gen", "tipo_celular", "log2FC"]].to_dict())

patron_pos = meta_summary.groupby("tipo_celular")["log2FC"].apply(lambda x: (x > 0).sum()).rename("sobreexpresados")
patron_neg = meta_summary.groupby("tipo_celular")["log2FC"].apply(lambda x: (x < 0).sum()).rename("infraexpresados")
print("\nPatron sobre/infraexpresion por tipo celular:")
print(pd.concat([patron_pos, patron_neg], axis=1).sort_values("sobreexpresados", ascending=False))

Total DEGs: 5080

Tipo celular con mas DEGs:
tipo_celular
Colonocytes                         1124
Monocyte (subtipo indeterminado)     922
PC IgA                               850
PC IgG                               637
CD4                                  545
dtype: int64

Desglose por comparacion dentro de Colonocytes:
comparacion
UC_vs_HC    685
CD_vs_HC    383
UC_vs_CD     56
dtype: int64

--- UC_vs_HC ---
Mas sobreexpresado: {'gen': 'REG3A', 'tipo_celular': 'Goblet', 'log2FC': 16.525}
Mas infraexpresado: {'gen': 'MT1H', 'tipo_celular': 'Colonocytes', 'log2FC': -6.6039}

--- CD_vs_HC ---
Mas sobreexpresado: {'gen': 'REG3A', 'tipo_celular': 'Goblet', 'log2FC': 14.508}
Mas infraexpresado: {'gen': 'CYP3A4', 'tipo_celular': 'Colonocytes', 'log2FC': -6.6106}

Patron sobre/infraexpresion por tipo celular:
                                    sobreexpresados  infraexpresados
tipo_celular                                                        
Colonocytes                                  

# · Features para ML



In [ ]:
long_rows = []

for (ct, comparison), sig_df in deg_tables.items():
    if sig_df.empty:
        continue
    dds = dds_by_ct[ct]
    normed = pd.DataFrame(dds.layers['normed_counts'], index=dds.obs_names, columns=dds.var_names)
    normed_log = np.log2(normed + 1)

    for _, row in sig_df.iterrows():
        gene = row['gene']
        if gene not in normed_log.columns:
            continue
        for patient_id, value in normed_log[gene].items():
            long_rows.append({
                'paciente': patient_id,
                'tipo_celular': ct,
                'comparacion': comparison,
                'gen': gene,
                'valor_expresion_normalizada': value,
                'log2FC': row['log2FoldChange'],
                'padj': row['padj'],
            })

long_df = pd.DataFrame(long_rows)
long_path = f"{TABLES_DIR}/features_long_table.csv"
long_df.to_csv(long_path, index=False, float_format='%.5g')
print(f"Tabla larga: {long_df.shape} → {Path(long_path).name}")


Tabla larga: (89091, 7) → features_long_table.csv


In [ ]:
# ── Pivotar a tabla ancha: 1 fila por paciente, columnas {tipo}__{gen}__{comparacion} ──
if not long_df.empty:
    long_df['column_name'] = (
        long_df['tipo_celular'].str.replace(' ', '_') + '__' +
        long_df['gen'] + '__' + long_df['comparacion']
    )

    wide_df = long_df.pivot_table(
        index='paciente', columns='column_name', values='valor_expresion_normalizada',
        aggfunc='first'
    )
    # No imputar NaN: si a un paciente le falta pseudobulk válido para ese tipo celular
    # (no pasó el umbral de 10 células), se deja NaN — Notebook 09 decide qué hacer.
    wide_df = wide_df.reset_index()

    wide_path = f"{TABLES_DIR}/features_wide_table.csv"
    wide_df.to_csv(wide_path, index=False, float_format='%.5g')
    print(f"Tabla ancha: {wide_df.shape} → {Path(wide_path).name}")

    # ── Metadatos de columnas: tipo celular, comparación, log2FC, padj ──
    col_metadata = (long_df[['column_name', 'tipo_celular', 'gen', 'comparacion', 'log2FC', 'padj']]
                     .drop_duplicates(subset='column_name'))
    meta_path = f"{TABLES_DIR}/features_wide_table_metadata.csv"
    col_metadata.to_csv(meta_path, index=False, float_format='%.5g')
    print(f"Metadatos de columnas: {col_metadata.shape} → {Path(meta_path).name}")
else:
    print("  No hay genes significativos en ninguna comparación — no se generan tablas de features.")

# No se guardan estas tablas en adata.obsm


Tabla ancha: (18, 5081) → features_wide_table.csv
Metadatos de columnas: (5080, 6) → features_wide_table_metadata.csv


# · Control de calidad

In [ ]:
# ── PCA de muestras pseudobulk (por tipo celular, sobre valores normalizados) ──
n_ct_plot = len(dds_by_ct)
if n_ct_plot > 0:
    ncols = 3
    nrows = int(np.ceil(n_ct_plot / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows))
    axes = np.atleast_1d(axes).flatten()

    for i, (ct, dds) in enumerate(dds_by_ct.items()):
        normed = pd.DataFrame(dds.layers['normed_counts'], index=dds.obs_names, columns=dds.var_names)
        log_normed = np.log2(normed + 1)
        # Top genes más variables de este pseudobulk para el PCA (criterio propio, no HVG heredado)
        top_var_genes = log_normed.var(axis=0).sort_values(ascending=False).head(min(2000, log_normed.shape[1])).index
        X = log_normed[top_var_genes].values
        X = X - X.mean(axis=0)
        try:
            U, S, Vt = np.linalg.svd(X, full_matrices=False)
            pcs = U[:, :2] * S[:2]
        except np.linalg.LinAlgError:
            continue

        ax = axes[i]
        conds = dds.obs[CONDITION_KEY].values
        for cond in CONDITIONS:
            mask = conds == cond
            if mask.any():
                ax.scatter(pcs[mask, 0], pcs[mask, 1], label=cond, color=PALETTE[cond], s=60)
        ax.set_title(ct, fontsize=9)
        ax.set_xlabel("PC1"); ax.set_ylabel("PC2")
        ax.legend(fontsize=6)

    for j in range(i + 1, len(axes)):
        axes[j].axis('off')

    plt.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/pca_pseudobulk_by_celltype.png", dpi=130, bbox_inches='tight', facecolor='white')
    plt.close(fig)
    print("   → Figura guardada: pca_pseudobulk_by_celltype.png")


   → Figura guardada: pca_pseudobulk_by_celltype.png


In [ ]:
# ── Heatmap de correlación entre muestras (por tipo celular) ────────────
for ct, dds in dds_by_ct.items():
    normed = pd.DataFrame(dds.layers['normed_counts'], index=dds.obs_names, columns=dds.var_names)
    log_normed = np.log2(normed + 1)
    corr = log_normed.T.corr()

    fig, ax = plt.subplots(figsize=(0.6 * len(corr) + 2, 0.6 * len(corr) + 2))
    sns.heatmap(corr, cmap='RdBu_r', vmin=-1, vmax=1, square=True,
                xticklabels=True, yticklabels=True, ax=ax,
                cbar_kws={'label': 'Correlación de Pearson'})
    ax.set_title(f"Correlación entre pseudobulks — {ct}", fontsize=10)
    plt.tight_layout()
    ct_safe = ct.replace(' ', '_').replace('/', '-')
    fig.savefig(f"{FIGURES_DIR}/corr_heatmap_{ct_safe}.png", dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)

print(f"   → {len(dds_by_ct)} heatmaps de correlación guardados (uno por tipo celular)")


   → 14 heatmaps de correlación guardados (uno por tipo celular)


In [ ]:
# ── Histograma de p-valores + MA-plot por comparación ───────────────────
for (ct, comparison), ds in stats_by_ct_comparison.items():
    res = ds.results_df
    ct_safe = ct.replace(' ', '_').replace('/', '-')

    fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

    axes[0].hist(res['pvalue'].dropna(), bins=40, color='#2E86AB', edgecolor='white')
    axes[0].set_title(f"Histograma p-valores\n{ct} | {comparison}", fontsize=9)
    axes[0].set_xlabel("p-valor"); axes[0].set_ylabel("Nº genes")

    sig_mask = (res['padj'] < DEG_PADJ_MAX) & (res['log2FoldChange'].abs() > DEG_LOG2FC_MIN)
    axes[1].scatter(np.log10(res['baseMean'] + 1), res['log2FoldChange'],
                     s=4, alpha=0.3, color='grey', label='No sig.')
    axes[1].scatter(np.log10(res.loc[sig_mask, 'baseMean'] + 1), res.loc[sig_mask, 'log2FoldChange'],
                     s=6, alpha=0.7, color='#E84855', label='Sig.')
    axes[1].axhline(0, color='black', lw=0.8)
    axes[1].set_title(f"MA-plot\n{ct} | {comparison}", fontsize=9)
    axes[1].set_xlabel("log10(baseMean + 1)"); axes[1].set_ylabel("log2FoldChange")
    axes[1].legend(fontsize=7)

    plt.tight_layout()
    fig.savefig(f"{FIGURES_DIR}/qc_{ct_safe}_{comparison}.png", dpi=120, bbox_inches='tight', facecolor='white')
    plt.close(fig)

print(f"   → {len(stats_by_ct_comparison)} pares (histograma p-valor + MA-plot) guardados")


   → 40 pares (histograma p-valor + MA-plot) guardados


# · Resumen final

In [ ]:
summary = {
    'tipos_celulares_analizados': len(dds_by_ct),
    'comparaciones_completadas': len(stats_by_ct_comparison),
    'tipos_celulares_excluidos_por_condicion_unica': single_condition_ct,
    'genes_deg_significativos_total': len(all_deg),
    'design_por_tipo_celular': design_by_ct,
    'raw_source_usada': raw_source,
}

print("   NOTEBOOK 08 COMPLETADO")

for k, v in summary.items():
    print(f"  {k}: {v}")

import json
with open(f"{TABLES_DIR}/08_run_summary.json", 'w') as f:
    json.dump(summary, f, indent=2, default=str)
print(f"\n  → Resumen guardado: 08_run_summary.json")
